# 143 현재 고정 미성형 개발 후보: A OR B OR C OR D
A std24군집전문가, B 조건비교physical12, C shock6세시드평균+배럴5std95%, D 동시변화intensity4LR+호퍼냉각95%. C innernormal99%,D99.5%. 학습Run제외점수를기록하나 규칙 선택은본평가자료에서이루어져새자료검증은아님.

In [1]:
from pathlib import Path
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables'
a=pd.read_csv(t/'rg3_shortshot_133_predictions.csv',parse_dates=['TimeStamp']);a=a[a.model.eq('fixed_OR')][['TimeStamp','A','B']]
c=pd.read_csv(t/'rg3_shortshot_139_predictions.csv',parse_dates=['TimeStamp']);c=c[c.model.eq('primary_three_seed_mean')][['TimeStamp','run_id','type','actual','prediction','new','score','threshold','gate_value','gate_threshold']]
c=c.rename(columns={'prediction':'ABC_prediction','new':'C','score':'C_score','threshold':'C_threshold','gate_value':'C_barrel5_std','gate_threshold':'C_barrel5_std_threshold'})
d=pd.read_csv(t/'rg3_shortshot_141_predictions.csv',parse_dates=['TimeStamp']);d=d[d.model.eq('hopper_cool_0.95_budget0.005')][['TimeStamp','prediction','new','score']].rename(columns={'prediction':'saved_prediction','new':'D','score':'D_score'})
z=c.merge(a,on='TimeStamp',validate='one_to_one').merge(d,on='TimeStamp',validate='one_to_one')
z['prediction']=z.A|z.B|z.C|z.D
assert (z.prediction==z.saved_prediction).all()
assert len(z)==591 and z.TimeStamp.is_unique
rows=[]
for cohort,mask in [('normal_and_shortshot',z.type.isin(['정상','미성형'])),('all_591_shortshot_target',np.ones(len(z),bool))]:
 q=z[mask];tn,fp,fn,tp=confusion_matrix(q.actual,q.prediction,labels=[0,1]).ravel()
 rows.append(dict(cohort=cohort,N=len(q),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
cases=z[z.type.eq('미성형')]
byrun=[]
for run,q in z[z.type.isin(['정상','미성형'])].groupby('run_id'):
 tn,fp,fn,tp=confusion_matrix(q.actual,q.prediction,labels=[0,1]).ravel()
 byrun.append(dict(run_id=run,normal=int(q.type.eq('정상').sum()),shortshot=int(q.actual.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn)))
for obj,name in [(z,'predictions'),(m,'metrics'),(cases,'all8_cases'),(pd.DataFrame(byrun),'byrun'),(z[z.type.eq('정상')&z.prediction.eq(1)],'false_positives')]:obj.to_csv(t/f'rg3_shortshot_143_fixed_candidate_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False));print(cases[['TimeStamp','run_id','A','B','C','D','prediction']].to_string(index=False))
print(pd.DataFrame(byrun).to_string(index=False))


                  cohort   N  TP  FP  FN  TN  precision  recall   F1
    normal_and_shortshot 574   6  11   2 555   0.352941    0.75 0.48
all_591_shortshot_target 591   6  11   2 572   0.352941    0.75 0.48
          TimeStamp run_id  A  B  C  D  prediction
2020-10-22 01:09:23 rg3_R2  0  0  0  0           0
2020-10-22 04:48:39 rg3_R3  1  0  0  0           1
2020-10-22 05:42:13 rg3_R3  0  0  0  1           1
2020-10-22 06:12:03 rg3_R3  1  0  0  0           1
2020-10-22 06:58:22 rg3_R3  0  1  0  0           1
2020-10-23 02:25:54 rg3_R4  0  0  0  0           0
2020-10-23 05:47:40 rg3_R5  0  0  1  0           1
2020-10-23 05:54:52 rg3_R5  0  0  1  0           1
run_id  normal  shortshot  TP  FP  FN  TN
rg3_R1      41          0   0   0   0  41
rg3_R2      98          1   0   2   1  96
rg3_R3     187          4   4   5   0 182
rg3_R4      70          1   0   0   1  70
rg3_R5     170          2   2   4   0 166
